# 🤖 ROTBOTS — Script Generator
## From Sources to Storyboard

**Workshop: "Anatomy of a Content Machine"**  
LI-MA Transformation Digital Art 2026, Amsterdam

---

This notebook takes you through the first half of the ROTBOTS pipeline:

1. **Feed the Machine** — Paste URLs or text, the system scrapes and summarizes them
2. **The Script Writer** — An LLM generates a structured essay with narration and visual directions
3. **Visual Translation** — The essay becomes a storyboard with styles and text-to-video prompts

The output (`prompts.json`) will be used in the next notebook for video generation.

**You don't need to understand the code.** Just fill in your inputs and press ▶️ Play on each cell.

---

## 🔧 Station 0: Setup

This connects to Google Drive so your results are saved and available to the other notebooks.

In [ ]:
# ============================================================
# SETUP — Run this cell first!
# ============================================================

!pip install -q requests beautifulsoup4 pymupdf

import json
import re
import random
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, JSON, HTML

# Connect to Google Drive (shared workspace between notebooks)
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Dependencies installed")
print(f"📁 Shared workspace: {WORK_DIR} (on Google Drive)")

In [ ]:
# ============================================================
# API KEY — Paste your Groq API key here
# Get one for free at: https://console.groq.com/keys
# ============================================================

GROQ_API_KEY = ""  # <-- Paste your Groq API key here (starts with gsk_)

# LLM Settings — don't change these unless you know what you're doing
LLM_MODEL = "llama-3.3-70b-versatile"  # Free on Groq
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"  # DO NOT put your key here!

if not GROQ_API_KEY:
    print("⚠️  Please paste your Groq API key above!")
    print("   Get one free at: https://console.groq.com/keys")
elif not GROQ_API_KEY.startswith("gsk_"):
    print("⚠️  That doesn't look like a Groq API key (should start with gsk_)")
    print("   Make sure you pasted it in the GROQ_API_KEY field, not somewhere else!")
else:
    print(f"✅ API key configured (model: {LLM_MODEL})")

In [ ]:
# ============================================================
# HELPER FUNCTIONS — The engine under the hood
# You don't need to modify this cell, just run it.
# ============================================================

def query_llm(prompt, system_prompt=None, temperature=None):
    """Send a prompt to the Groq LLM API."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": temperature or LLM_TEMPERATURE,
        "max_tokens": LLM_MAX_TOKENS,
    }
    response = requests.post(
        GROQ_API_URL,
        headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"]
    # Clean thinking blocks from models that use them
    if "<think>" in content and "</think>" in content:
        content = content.split("</think>")[-1].strip()
    return content


def parse_json_response(response):
    """Robust JSON parser for LLM output. Handles markdown fences, trailing
    commas, unescaped characters, truncated output, and other common issues."""
    response = response.strip()
    # Strip control characters
    response = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', response)

    # Strip markdown code fences
    if response.startswith("```"):
        lines = response.split("\n")
        if lines[-1].strip() == "```":
            response = "\n".join(lines[1:-1])
        else:
            response = "\n".join(lines[1:])
    response = response.strip()

    # Find the start of JSON
    if not response.startswith("[") and not response.startswith("{"):
        json_start = -1
        for ch in ['{', '[']:
            idx = response.find(ch)
            if idx != -1 and (json_start == -1 or idx < json_start):
                json_start = idx
        if json_start != -1:
            response = response[json_start:]

    # Strategy 1: Find balanced JSON by tracking braces
    def find_balanced_json(text):
        if not text or text[0] not in '{[':
            return None
        open_char = text[0]
        close_char = '}' if open_char == '{' else ']'
        depth = 0
        in_string = False
        escape = False
        for i, ch in enumerate(text):
            if escape:
                escape = False
                continue
            if ch == '\\\\':
                escape = True
                continue
            if ch == '"' and not escape:
                in_string = not in_string
                continue
            if in_string:
                continue
            if ch == open_char:
                depth += 1
            elif ch == close_char:
                depth -= 1
                if depth == 0:
                    return text[:i+1]
        return None

    balanced = find_balanced_json(response)
    if balanced:
        # Try parsing the balanced JSON
        for attempt_text in [balanced, re.sub(r',\s*([}\\]])', r'\\1', balanced)]:
            try:
                return json.loads(attempt_text, strict=False)
            except json.JSONDecodeError:
                continue

    # Strategy 2: Try truncating at last } or ]
    for end_char in ['}', ']']:
        last_idx = response.rfind(end_char)
        if last_idx != -1:
            truncated = response[:last_idx + 1]
            # Try as-is, then with trailing commas removed
            for attempt_text in [truncated, re.sub(r',\s*([}\\]])', r'\\1', truncated)]:
                try:
                    return json.loads(attempt_text, strict=False)
                except json.JSONDecodeError:
                    continue

    # Strategy 3: Try to fix common issues and parse
    cleaned = response
    cleaned = re.sub(r',\s*([}\\]])', r'\\1', cleaned)  # trailing commas
    cleaned = re.sub(r'([{,])\s*(\w+)\s*:', r'\\1"\\2":', cleaned)  # unquoted keys
    try:
        return json.loads(cleaned, strict=False)
    except json.JSONDecodeError:
        pass

    # Strategy 4: Last resort — try the raw response
    return json.loads(response, strict=False)


def fetch_website_text(url, max_chars=10000):
    """Scrape main text content from a URL."""
    url = url.strip().rstrip('/').strip()
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"}, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for el in soup(['script', 'style', 'nav', 'header', 'footer', 'aside', 'form']):
        el.decompose()
    main_content = None
    for sel in ['article', 'main', '[role="main"]', '.content', '#content', '.post']:
        main_content = soup.select_one(sel)
        if main_content:
            break
    text = main_content.get_text(separator=' ', strip=True) if main_content else (soup.find('body') or soup).get_text(separator=' ', strip=True)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:max_chars] if len(text) > max_chars else text


def fetch_pdf_text(source, max_chars=10000):
    """Extract text from a PDF (URL or local file path)."""
    import tempfile
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    pdf_path = None
    temp_file = None
    try:
        if source.startswith("http"):
            resp = requests.get(source, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
            resp.raise_for_status()
            temp_file = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
            temp_file.write(resp.content)
            temp_file.close()
            pdf_path = temp_file.name
        else:
            pdf_path = source
        doc = fitz.open(pdf_path)
        text_parts = []
        total_chars = 0
        for page in doc:
            page_text = page.get_text()
            text_parts.append(page_text)
            total_chars += len(page_text)
            if total_chars >= max_chars:
                break
        doc.close()
        text = "\n".join(text_parts)
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text[:max_chars] if len(text) > max_chars else text
    finally:
        if temp_file:
            import os
            os.unlink(temp_file.name)


print("✅ Helper functions loaded")

---
## 📥 Station 1: Feed the Machine

Paste your sources below. The machine will scrape websites, read PDFs, and accept raw text.

> **🤔 Think about:** What gets lost in this compression? What does the machine decide is "important"?

In [ ]:
# ============================================================
# YOUR INPUTS — Edit these!
# ============================================================

TOPIC = "The history and ethics of AI-generated art"

SOURCES = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence_art",
    # "https://example.com/another-article",
    # "You can also just paste raw text here as a string.",
]

In [ ]:
# SCRAPE SOURCES
print(f"📥 Processing {len(SOURCES)} source(s) for topic: '{TOPIC}'")
print("=" * 60)
source_texts = []
for i, source in enumerate(SOURCES):
    print(f"\n🔗 Source {i+1}: {source[:80]}..." if len(source) > 80 else f"\n🔗 Source {i+1}: {source}")
    try:
        if source.startswith("http") and source.lower().endswith(".pdf"):
            print("   Type: PDF (URL)"); text = fetch_pdf_text(source)
        elif source.startswith("http"):
            print("   Type: Website"); text = fetch_website_text(source)
        elif source.endswith(".pdf"):
            print("   Type: PDF (local)"); text = fetch_pdf_text(source)
        else:
            print("   Type: Raw text"); text = source
        print(f"   Extracted: {len(text)} characters")
        source_texts.append({"source": source[:100], "text": text})
    except Exception as e:
        print(f"   ❌ Error: {e}")
print(f"\n✅ {len(source_texts)} source(s) loaded")

In [ ]:
# SUMMARIZE SOURCES
print("🧠 Summarizing sources with LLM...")
summaries = []
for i, src in enumerate(source_texts):
    print(f"\n📝 Summarizing source {i+1}/{len(source_texts)}...")
    summary = query_llm(
        f'Summarize this text in 3-5 paragraphs for a video essay about "{TOPIC}":\n\n{src["text"][:6000]}',
        system_prompt="You are a research assistant. Summarize accurately and concisely."
    )
    summaries.append({"source": src["source"], "summary": summary})
    print(f"   ✅ Done ({len(summary)} chars)")

with open(WORK_DIR / "summaries.json", "w") as f:
    json.dump({"topic": TOPIC, "sources": summaries}, f, indent=2)
print(f"\n✅ Summaries saved to Google Drive")

In [ ]:
# VIEW SUMMARIES
for i, s in enumerate(summaries):
    display(Markdown(f"### Source {i+1}: {s['source']}"))
    display(Markdown(s['summary']))
    display(Markdown("---"))

---
## ✍️ Station 2: The Script Writer

> **🤔 Think about:** Who is the "author" here? You chose the topic, but the LLM decided the argument, the structure, and the words.

In [ ]:
# SCRIPT SETTINGS
NUM_CHAPTERS = 2
SECTIONS_PER_CHAPTER = 2
print(f"📐 {NUM_CHAPTERS} chapters × {SECTIONS_PER_CHAPTER} sections = {NUM_CHAPTERS * SECTIONS_PER_CHAPTER} content scenes + title + credits")

In [ ]:
# GENERATE ESSAY OUTLINE
print("🧠 Generating essay outline...")
summaries_text = "\n\n".join([f"SOURCE: {s['source']}\n{s['summary']}" for s in summaries])
outline_prompt = f'Create an essay outline for a video essay about: "{TOPIC}"\n\nAVAILABLE RESEARCH:\n{summaries_text}\n\nREQUIREMENTS:\n- Create exactly {NUM_CHAPTERS} chapters.\n- Each chapter should present a distinct argument\n- Build a logical progression\n\nFormat as JSON:\n{{"title": "...", "thesis": "...", "chapters": [{{"chapter": 1, "title": "...", "summary": "...", "key_points": ["..."]}}]}}\n\nOnly output the JSON.'
for attempt in range(3):
    try:
        outline = parse_json_response(query_llm(outline_prompt, "You are an expert essay writer. Create compelling video essay scripts."))
        break
    except Exception as e:
        print(f"   Attempt {attempt+1}/3 failed: {e}")
        if attempt == 2: raise
if len(outline.get("chapters", [])) > NUM_CHAPTERS: outline["chapters"] = outline["chapters"][:NUM_CHAPTERS]
print(f"\n📖 Title: {outline.get('title', 'Untitled')}")
print(f"💡 Thesis: {outline.get('thesis', '')}")
for ch in outline.get("chapters", []): print(f"   Ch {ch['chapter']}: {ch['title']}")

In [ ]:
# WRITE CHAPTERS
print("✍️ Writing chapters...")
chapter_system = "You are an expert video essay scriptwriter. Write detailed narration (3-6 sentences per section) with visual directions and mood."
for i, chapter in enumerate(outline["chapters"]):
    print(f"\n📝 Chapter {chapter['chapter']}: {chapter['title']}")
    chapter_prompt = f'Write sections for:\nTOPIC: {TOPIC}\nTHESIS: {outline.get("thesis", "")}\nCHAPTER {chapter["chapter"]}: {chapter["title"]}\nSummary: {chapter.get("summary", "")}\nKey points: {json.dumps(chapter.get("key_points", []))}\nCONTEXT:\n{summaries_text[:4000]}\n\nWrite exactly {SECTIONS_PER_CHAPTER} sections as JSON array:\n[{{"section": 1, "narration": "...", "visual_direction": "...", "mood": "..."}}]\n\nOnly output JSON.'
    try:
        sections = parse_json_response(query_llm(chapter_prompt, chapter_system))
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except:
        sections = [{"section": 1, "narration": chapter["title"], "visual_direction": chapter.get("summary", ""), "mood": "neutral"}]
    if len(sections) > SECTIONS_PER_CHAPTER: sections = sections[:SECTIONS_PER_CHAPTER]
    outline["chapters"][i]["sections"] = sections
    for s in sections: print(f"   Section {s.get('section','?')}: {s.get('narration','')[:80]}...")
outline["credits"] = {"sources": [s["source"] for s in summaries]}
with open(WORK_DIR / "essay_script.json", "w") as f: json.dump(outline, f, indent=2)
print(f"\n✅ Essay saved to Google Drive")

In [ ]:
# VIEW ESSAY
display(Markdown(f"# {outline.get('title', 'Untitled')}"))
display(Markdown(f"*{outline.get('thesis', '')}*\n\n---"))
for ch in outline["chapters"]:
    display(Markdown(f"## Chapter {ch['chapter']}: {ch['title']}"))
    for s in ch.get("sections", []):
        display(Markdown(f"**Section {s.get('section','?')}** — *{s.get('mood','?')}*\n\n> 🎙️ {s.get('narration','')}\n\n> 🎬 {s.get('visual_direction','')}\n\n---"))

---
## 🎬 Station 3: Visual Translation

> **🤔 Think about:** The machine decides what "documentary" or "horror" looks like. What assumptions are baked into these categories?

In [ ]:
# CHOOSE STYLE ARC
STYLE_ARCS = {
    "calm_to_dramatic": {"description": "Starts calm, builds to intense", "early": ["documentary", "nature"], "middle": ["news_report", "sports_commentary"], "late": ["action_movie", "horror"], "credits": ["documentary"]},
    "documentary_journey": {"description": "Classic documentary with increasing energy", "early": ["documentary"], "middle": ["nature", "news_report", "documentary"], "late": ["action_movie", "sports_commentary"], "credits": ["documentary"]},
    "chaos_arc": {"description": "Chaotic brainrot energy, escalating surrealism", "early": ["classic_brainrot", "zoomer_brainrot"], "middle": ["surreal_brainrot", "hyperpop_brainrot"], "late": ["fever_dream_brainrot", "cursed_brainrot"], "credits": ["documentary"]},
}
CONTENT_STYLES = {
    "documentary": {"visual_keywords": "cinematic, professional lighting, steady shots", "audio_keywords": "ambient sounds, subtle music"},
    "nature": {"visual_keywords": "wide nature shots, golden hour, landscapes", "audio_keywords": "nature sounds, bird calls, wind"},
    "news_report": {"visual_keywords": "news studio aesthetic, professional framing", "audio_keywords": "news broadcast audio, serious tone"},
    "sports_commentary": {"visual_keywords": "dynamic angles, action shots, slow motion", "audio_keywords": "crowd cheering, energetic"},
    "action_movie": {"visual_keywords": "dynamic movement, dramatic lighting, wide shots", "audio_keywords": "orchestral hits, impacts"},
    "horror": {"visual_keywords": "dark lighting, shadows, dutch angles, fog", "audio_keywords": "creepy ambience, ominous drones"},
    "classic_brainrot": {"visual_keywords": "chaotic editing, deep fried aesthetic", "audio_keywords": "bass boosted, vine booms"},
    "surreal_brainrot": {"visual_keywords": "dreamlike, impossible geometry", "audio_keywords": "slowed reverb, distorted"},
    "zoomer_brainrot": {"visual_keywords": "meme aesthetic, ironic", "audio_keywords": "TikTok sounds, bass boosted"},
    "hyperpop_brainrot": {"visual_keywords": "maximalist, Y2K, chrome", "audio_keywords": "hyperpop beats, synth stabs"},
    "fever_dream_brainrot": {"visual_keywords": "hallucinatory, warped", "audio_keywords": "echoing, time-stretched"},
    "cursed_brainrot": {"visual_keywords": "unsettling, jpeg artifacts", "audio_keywords": "distorted, ominous"},
    "none": {"visual_keywords": "", "audio_keywords": ""}
}

CHOSEN_ARC = "calm_to_dramatic"  # Options: calm_to_dramatic, documentary_journey, chaos_arc
arc = STYLE_ARCS[CHOSEN_ARC]
print(f"🎨 Style Arc: {CHOSEN_ARC} — {arc['description']}")

In [ ]:
# STORYBOARD + STYLE ASSIGNMENT
print("🎬 Generating storyboard...")
scenes = []; scene_num = 1
scenes.append({"scene": scene_num, "scene_type": "title_card", "title": outline.get("title", "Untitled"), "description": outline.get("thesis", "")}); scene_num += 1
for ch in outline.get("chapters", []):
    for sec in ch.get("sections", []):
        scenes.append({"scene": scene_num, "scene_type": "content", "title": f"{ch['title']} - Part {sec.get('section',1)}", "narration": sec.get("narration",""), "visual_direction": sec.get("visual_direction",""), "mood": sec.get("mood",""), "chapter": ch.get("chapter",0)}); scene_num += 1
scenes.append({"scene": scene_num, "scene_type": "credits", "title": "Credits", "description": ", ".join(outline.get("credits",{}).get("sources",[]))})
content_scenes = [s for s in scenes if s["scene_type"]=="content"]
total=len(content_scenes); early_end=max(1,int(total*0.25)); late_start=max(early_end+1,int(total*0.75))
last_style=None
for i,sc in enumerate(content_scenes):
    pool = arc.get("early" if i<early_end else "late" if i>=late_start else "middle", ["documentary"])
    available=[s for s in pool if s!=last_style] or pool; style=random.choice(available)
    sc["assigned_style"]=style; sc["visual_keywords"]=CONTENT_STYLES.get(style,{}).get("visual_keywords",""); sc["audio_keywords"]=CONTENT_STYLES.get(style,{}).get("audio_keywords",""); last_style=style
with open(WORK_DIR/"storyboard.json","w") as f: json.dump(scenes,f,indent=2)
for s in scenes:
    tag=f" [{s.get('assigned_style','')}]" if s.get('assigned_style') else ""
    print(f"   Scene {s['scene']}: [{s['scene_type']}] {s['title']}{tag}")

In [ ]:
# GENERATE T2V PROMPTS
print("🎥 Converting to video prompts...")
OPENINGS = ["Start with SHOT TYPE","Start with ACTION","Start with ENVIRONMENT","Start with LIGHTING","Start with CAMERA MOVEMENT","Start with ATMOSPHERE"]
prompts = []
for sc in scenes:
    if sc["scene_type"]!="content": continue
    style=sc.get("assigned_style","none"); vk=sc.get("visual_keywords",""); ak=sc.get("audio_keywords","")
    style_inst=f"\nStyle: {style}\nVisual: {vk}\nAudio: {ak}" if style!="none" else ""
    sys_p=f"You are an expert at text-to-video prompts. Write ONE flowing paragraph, 4-8 sentences: shot type, setting, action, camera, audio.{style_inst}\nRULES: No text/writing. Include audio. Max 2 subjects."
    user_p=f"Convert to T2V prompt:\nVisual: {sc.get('visual_direction','')}\nMood: {sc.get('mood','')}\nTitle: {sc.get('title','')}\n\n{random.choice(OPENINGS)}\n\nOutput ONLY the prompt."
    print(f"   Scene {sc['scene']}: {sc['title']} [{style}]...", end=" ")
    t2v=query_llm(user_p,sys_p).strip().strip('"')
    prompts.append({"scene":sc["scene"],"title":sc["title"],"t2v_prompt":t2v,"style":style,"narration":sc.get("narration",""),"duration":5})
    print(f"✅")
with open(WORK_DIR/"prompts.json","w") as f: json.dump(prompts,f,indent=2)
print(f"\n✅ {len(prompts)} prompts saved to Google Drive")

In [ ]:
# VIEW FINAL PROMPTS
display(Markdown(f"# 🎬 Video Production Plan\n*{len(prompts)} scenes × ~5s = ~{len(prompts)*5}s of video*\n\n---"))
for p in prompts:
    display(Markdown(f"### Scene {p['scene']}: {p['title']}\n**Style:** `{p['style']}` | **Duration:** {p['duration']}s\n\n> 🎙️ {p['narration']}\n\n> 🎥 {p['t2v_prompt']}\n\n---"))
print("\n📁 All files saved to Google Drive/rotbots_workshop/")
print("   → Open 05_Generate.ipynb next to create videos from these prompts!")

---
## ⏭️ Next Steps

Your `prompts.json` is saved on Google Drive. Open **05_Generate.ipynb** — it will find the file automatically.

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026*